# 🤖 Robot Fleet Monitoring — Nostradamus IoTO SDK

Warehouse robot fleet: AGVs, robotic arms and an inspection drone report position, velocity, orientation, battery, motor temperature, payload, task status, LiDAR point count, network latency, CPU usage and error codes.

This notebook mirrors `examples/cyber_physical/robot_fleet_monitoring.py` using only `nostradamus_ioto_sdk.NostradamusClient`.
It sends **5 robots × 24 hourly readings** (120 records) to the collection
`robot_fleet`, then queries, aggregates and deletes part of it.

## Workflow

1. Create the `robot_fleet` collection with a schema
2. Generate and send 5 robots × 24 hourly readings
3. Query errors, low battery, performance bottlenecks and battery ranking
4. Aggregate battery, motor temperature and obstacle-distance statistics
5. Delete a 12-hour window for `AGV_001`
6. Delete the collection

## What this notebook highlights

- Mixed robot types (AGV, robotic arm, inspection drone)
- Error detection (`error_code > 0`) and task-status inspection
- Performance bottlenecks (high CPU with high network latency)

## Prerequisites

```bash
pip install -e ../..          # from the repository root
pip install python-dotenv
```

| Variable | Purpose |
|----------|---------|
| `NOSTRADAMUS_PROJECT_ID` | Target project UUID |
| `NOSTRADAMUS_MASTER_KEY` | Master key (create/delete collection, delete data) |
| `NOSTRADAMUS_WRITE_KEY` | Write key (send data) |
| `NOSTRADAMUS_READ_KEY` | Read key (query data and statistics) |
| `NOSTRADAMUS_BASE_URL` | Optional API base URL override |

> The notebook is idempotent: if the collection already exists the SDK raises
> `APIError` with status `409`, and Step 1 falls back to reusing it.

In [ ]:
%pip install -q -e ../.. python-dotenv

## Configuration

Each permission level gets its own `NostradamusClient` instance — the SDK injects
the `X-API-Key` header, so there is no manual header handling.

In [ ]:
import os
import random
from collections import Counter
from datetime import datetime, timedelta, timezone

from dotenv import load_dotenv

from nostradamus_ioto_sdk import NostradamusClient, StatOperation
from nostradamus_ioto_sdk.exceptions import APIError

load_dotenv()

BASE_URL = os.getenv("NOSTRADAMUS_BASE_URL", "https://nostradamus-ioto.issel.ee.auth.gr")
PROJECT_ID = os.getenv("NOSTRADAMUS_PROJECT_ID", "")
MASTER_KEY = os.getenv("NOSTRADAMUS_MASTER_KEY", "")
WRITE_KEY = os.getenv("NOSTRADAMUS_WRITE_KEY", "")
READ_KEY = os.getenv("NOSTRADAMUS_READ_KEY", "")

_missing = [
    name
    for name, value in {
    "PROJECT_ID": PROJECT_ID,
    "MASTER_KEY": MASTER_KEY,
    "WRITE_KEY": WRITE_KEY,
    "READ_KEY": READ_KEY,
    }.items()
    if not value
]
if _missing:
    raise RuntimeError(
        "Missing required environment variables: " + ", ".join(_missing)
    )

print("🤖 Nostradamus IoTO SDK — Robot Fleet Monitoring")
print(f"base_url={BASE_URL} project_id={PROJECT_ID}")

## Step 1: Create the robot collection

`collections.create()` returns a `CollectionResponse`, so the collection ID comes
straight from the model. The schema declares one sample record and is copied
verbatim from `examples/cyber_physical/robot_fleet_monitoring.py`.

In [ ]:
COLLECTION_NAME = "robot_fleet"

COLLECTION_SCHEMA = {
                    "key": "AGV_001",
                    "timestamp": "2025-06-17T10:30:00Z",
                    "robot_type": "agv",
                    "pos_x_m": 50.0,
                    "pos_y_m": 25.0,
                    "pos_z_m": 0.0,
                    "velocity_ms": 2.1,
                    "orientation_deg": 180,
                    "battery_pct": 85.0,
                    "motor_temperature_c": 42.0,
                    "payload_kg": 250.0,
                    "task_id": "MOVE_0042",
                    "task_status": "executing",
                    "obstacle_distance_m": 3.5,
                    "lidar_points_count": 25000,
                    "network_latency_ms": 15.0,
                    "cpu_usage_pct": 55.0,
                    "error_code": 0,
                }

with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:
    try:
        collection = master_client.collections.create(
            project_id=PROJECT_ID,
            name=COLLECTION_NAME,
            description="Warehouse robot fleet telemetry",
            tags=["robots", "warehouse", "cyber-physical"],
            collection_schema=COLLECTION_SCHEMA,
        )
        print("✅ Collection created")
    except APIError as exc:
        if exc.status_code != 409:
            raise
        collection = next(
            (
                coll
                for coll in master_client.collections.list(PROJECT_ID)
                if coll.collection_name == COLLECTION_NAME
            ),
            None,
        )
        if collection is None:
            raise
        print("⚠️  Collection already exists — reusing it")

collection_id = str(collection.collection_id)
print(f"collection_id={collection_id}")
collection.model_dump()

## Step 2: Generate and send robot fleet data

`data.send()` accepts a single dict or a list of dicts — the SDK serialises and
authenticates the batch with the write key. `generate_robot_data()` is the
same generator used by the script.

In [ ]:
ROBOT_PROFILES = {
    "AGV_001": {"type": "agv", "max_vel": 3.0, "max_payload": 500},
    "AGV_002": {"type": "agv", "max_vel": 3.0, "max_payload": 500},
    "ARM_001": {"type": "robotic_arm", "max_vel": 1.0, "max_payload": 25},
    "ARM_002": {"type": "robotic_arm", "max_vel": 1.0, "max_payload": 25},
    "DRONE_001": {"type": "inspection_drone", "max_vel": 5.0, "max_payload": 5},
}

TASK_PREFIXES = {
    "agv": "MOVE",
    "robotic_arm": "PICK",
    "inspection_drone": "INSPECT",
}

def generate_robot_data(num_readings=24):
    """Generate warehouse robot fleet telemetry data.

    Args:
        num_readings: Hourly readings per robot over 24h

    Returns:
        List of sensor readings
    """
    data = []
    base_time = datetime.now() - timedelta(hours=24)

    for robot_id, profile in ROBOT_PROFILES.items():
        rtype = profile["type"]
        max_vel = profile["max_vel"]
        max_payload = profile["max_payload"]
        battery = 95.0 + random.uniform(-5, 5)
        task_counter = random.randint(1, 50)

        for hour in range(num_readings):
            timestamp = base_time + timedelta(hours=hour)

            is_active = 6 <= timestamp.hour <= 22
            has_error = random.random() < 0.03

            if rtype == "agv":
                pos_x = random.uniform(5, 95)
                pos_y = random.uniform(5, 45)
                pos_z = 0.0
            elif rtype == "robotic_arm":
                pos_x = 20.0 + random.uniform(-2, 2)
                pos_y = 15.0 + random.uniform(-2, 2)
                pos_z = random.uniform(0, 2)
            else:
                pos_x = random.uniform(0, 100)
                pos_y = random.uniform(0, 50)
                pos_z = random.uniform(2, 8)

            velocity = (
                random.uniform(0.5, max_vel) if is_active else random.uniform(0, 0.3)
            )

            battery -= random.uniform(0.5, 2.0)
            if not is_active:
                battery += random.uniform(0.5, 1.5)
            battery = max(20, min(100, battery))

            payload = random.uniform(0, max_payload) if is_active else 0.0

            task_prefix = TASK_PREFIXES[rtype]
            task_counter += 1 if is_active else 0
            task_id = f"{task_prefix}_{task_counter:04d}"

            if has_error:
                task_status = "error"
            elif is_active:
                task_status = random.choice(
                    ["executing", "executing", "completed", "queued"]
                )
            else:
                task_status = "queued"

            motor_temp = 30 + velocity * 10 + random.uniform(-3, 5)

            reading = {
                "key": robot_id,
                "timestamp": timestamp.strftime("%Y-%m-%dT%H:%M:%S"),
                "robot_type": rtype,
                "pos_x_m": round(pos_x, 2),
                "pos_y_m": round(pos_y, 2),
                "pos_z_m": round(pos_z, 2),
                "velocity_ms": round(velocity, 2),
                "orientation_deg": random.randint(0, 359),
                "battery_pct": round(battery, 1),
                "motor_temperature_c": round(min(75, max(25, motor_temp)), 1),
                "payload_kg": round(payload, 1),
                "task_id": task_id,
                "task_status": task_status,
                "obstacle_distance_m": round(random.uniform(0.1, 10.0), 2),
                "lidar_points_count": random.randint(1000, 50000),
                "network_latency_ms": round(random.uniform(5, 100), 1),
                "cpu_usage_pct": round(random.uniform(10, 95), 1),
                "error_code": random.randint(1, 5) if has_error else 0,
            }
            data.append(reading)

    return data

data = generate_robot_data()
print(f"Generated {len(data)} records")

with NostradamusClient(api_key=WRITE_KEY, base_url=BASE_URL) as write_client:
    write_client.data.send(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        data=data,
    )
print(f"✅ Sent {len(data)} records to {COLLECTION_NAME}")

## Step 3: Query data with domain filters

`data.get()` returns a `DataListResponse` with `.data` (records) and
`.total_count`. Filters are plain dicts combined as an AND. The queries below
mirror the alert conditions from the script:

- **🤖 AGV_001 readings**
- **🚨 Errors (error_code > 0)**
- **🔋 Low battery (< 30 %)**
- **🐌 Bottlenecks (CPU > 80 % and latency > 50 ms)**
- **Lowest 5 battery levels**

In [ ]:
def summary(result, label):
    """Print a compact view of a DataListResponse."""
    print(f"{label}: {len(result.data)} of {result.total_count} record(s)")
    if result.data:
        print("  example:", result.data[0])


with NostradamusClient(api_key=READ_KEY, base_url=BASE_URL) as read_client:
    # All records, grouped per key
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
    )
    summary(result, "📥 All records")
    counts = Counter(row["key"] for row in result.data)
    print("   per key:", dict(sorted(counts.items())))

    # 🤖 AGV_001 readings
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "key", "operator": "eq", "property_value": "AGV_001"},
        ],
    )
    summary(result, "🤖 AGV_001 readings")

    # 🚨 Errors (error_code > 0)
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "error_code", "operator": "gt", "property_value": 0},
        ],
    )
    summary(result, "🚨 Errors (error_code > 0)")

    # 🔋 Low battery (< 30 %)
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "battery_pct", "operator": "lt", "property_value": 30},
        ],
    )
    summary(result, "🔋 Low battery (< 30 %)")

    # 🐌 Bottlenecks (CPU > 80 % and latency > 50 ms)
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "cpu_usage_pct", "operator": "gt", "property_value": 80},
            {"property_name": "network_latency_ms", "operator": "gt", "property_value": 50},
        ],
    )
    summary(result, "🐌 Bottlenecks (CPU > 80 % and latency > 50 ms)")

    # Lowest 5 battery levels
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        attributes=["key", "battery_pct", "robot_type"],
        order_by='{"field": "battery_pct", "order": "asc"}',
        limit=5,
    )
    print("🏆 Lowest 5 battery levels")
    for row in result.data:
        print("   >", row)

## Step 4: Statistics

`data.statistics()` takes `stat`, `attribute` and optional `group_by` /
`interval` / `limit`. Use the `StatOperation` enum instead of raw strings.

In [ ]:
with NostradamusClient(api_key=READ_KEY, base_url=BASE_URL) as read_client:
    for label, attribute, stat in [
        ("Average battery (%)", "battery_pct", StatOperation.AVG),
        ("Maximum motor temperature (°C)", "motor_temperature_c", StatOperation.MAX),
        ("Minimum obstacle distance (m)", "obstacle_distance_m", StatOperation.MIN),
        ("Distinct robots", "key", StatOperation.DISTINCT),
    ]:
        result = read_client.data.statistics(
            project_id=PROJECT_ID,
            collection_id=collection_id,
            stat=stat,
            attribute=attribute,
        )
        print(f"📈 {label}: {result.get('value', result)}")

## Step 5: Delete a 12-hour window for one robot

`data.delete()` accepts `key`, `timestamp_from` and `timestamp_to` (at least one
required). Here we remove the first half of the 24-hour window for
`AGV_001` and compare the record count before and after.

In [ ]:
timestamp_from = (datetime.now(timezone.utc) - timedelta(hours=24)).strftime(
    "%Y-%m-%dT%H:%M:%SZ"
)
timestamp_to = (datetime.now(timezone.utc) - timedelta(hours=12)).strftime(
    "%Y-%m-%dT%H:%M:%SZ"
)

DELETE_KEY = "AGV_001"


with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:

    def count_for(key):
        result = master_client.data.get(
            project_id=PROJECT_ID,
            collection_id=collection_id,
            filters=[{"property_name": "key", "operator": "eq", "property_value": key}],
        )
        return result.total_count

    before = count_for(DELETE_KEY)
    result = master_client.data.delete(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        key=DELETE_KEY,
        timestamp_from=timestamp_from,
        timestamp_to=timestamp_to,
    )
    after = count_for(DELETE_KEY)

print(f"🗑️  {result.message} (criteria={result.criteria})")
print(f"   {DELETE_KEY}: {before} → {after} record(s)")

## Step 6: Clean up

Remove the demo collection so the notebook can be re-run from scratch.

In [ ]:
with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:
    master_client.collections.delete(
        project_id=PROJECT_ID,
        collection_id=collection_id,
    )
print(f"🧹 Collection {collection_id} deleted")
print("✨ Demo complete!")